---
# 02. 전처리 — 퍼널 이벤트 정의 & 데이터 가공
---


01_eda에서 확인한 URL 구조를 기준으로 가입완료·이력서작성·지원완료 이벤트를 정의하고 유저 단위 퍼널 테이블을 생성한다. 이 노트북은 단독 실행할 수 있도록 DB 연결과 로그 로드를 포함한다.

## 1. 환경 설정 & 데이터 로드

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import koreanize_matplotlib

# notebooks/에서 실행해도 프로젝트 루트의 src 모듈을 불러올 수 있게 설정
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import get_engine, load_logs

warnings.filterwarnings("ignore")

engine = get_engine()
df = load_logs(engine)
df_filtered = df.copy()

## 2. Acquisition(가입 완료) 이벤트 정의

- signup/step3/done 외에 complete/github(깃허브 소셜 가입)도 가입완료로 포함시킴

In [ ]:
acquisition = (
    df_filtered[
        df_filtered['URL'].str.split('?').str[0].isin([
            'signup/step3/done',
            'complete/github'
        ])
    ]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'signup_time'})
)
print(f"Acquisition(가입 완료) 유저 수: {len(acquisition):,}명")


## 3. Activation ① — 이력서 작성 이벤트 정의 (step1 / step2)

In [ ]:
resume_step1 = (
    df_filtered[df_filtered['URL'].str.split('?').str[0].str.contains(
        'api/users/.+/resume/step1|@.+/resume/step1', regex=True, na=False
    )]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'resume_step1_time'})
)

resume_step2 = (
    df_filtered[df_filtered['URL'].str.split('?').str[0].str.contains(
        'api/users/.+/resume/step2|@.+/resume/step2', regex=True, na=False
    )]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'resume_step2_time'})
)


## 4. Activation ② — 지원 퍼널 이벤트 정의 (step1~4 / 완료)

In [ ]:
apply_steps = {}
for step in ['step1', 'step2', 'step3', 'step4']:
    apply_steps[step] = (
        df_filtered[df_filtered['URL'].str.split('?').str[0].isin([
            f'jobs/id/apply/{step}',
            f'api/jobs/id/apply/{step}'
        ])]
        [['user_uuid', 'timestamp']]
        .groupby('user_uuid')['timestamp'].min()
        .reset_index()
        .rename(columns={'timestamp': f'apply_{step}_time'})
    )

apply_complete = (
    df_filtered[df_filtered['URL'].str.split('?').str[0].isin([
        'jobs/id/apply/complete'
    ])]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'apply_complete_time'})
)


## 5. 유저 단위 퍼널 테이블 병합 및 전환 순서 검증

In [ ]:
funnel = acquisition.copy()
funnel = funnel.merge(resume_step1, on='user_uuid', how='left')
funnel = funnel.merge(resume_step2, on='user_uuid', how='left')
for step in ['step1', 'step2', 'step3', 'step4']:
    funnel = funnel.merge(apply_steps[step], on='user_uuid', how='left')
funnel = funnel.merge(apply_complete, on='user_uuid', how='left')

# 이전 단계보다 이후 시점에 발생한 경우에만 유효한 전환으로 인정
funnel['did_resume1']  = funnel['resume_step1_time']   >= funnel['signup_time']
funnel['did_resume2']  = funnel['resume_step2_time']   >= funnel['resume_step1_time']
funnel['did_apply1']   = funnel['apply_step1_time']    >= funnel['signup_time']
funnel['did_apply2']   = funnel['apply_step2_time']    >= funnel['apply_step1_time']
funnel['did_apply3']   = funnel['apply_step3_time']    >= funnel['apply_step2_time']
funnel['did_apply4']   = funnel['apply_step4_time']    >= funnel['apply_step3_time']
funnel['did_complete'] = funnel['apply_complete_time'] >= funnel['apply_step1_time']

print(f"퍼널 테이블 shape: {funnel.shape}")
funnel.dtypes


## 6. URL → 기능 카테고리 분류 함수

미전환 유저 마지막 행동이나 URL 트래픽을 기능별로 묶어볼 때 쓰는 함수

In [ ]:
def categorize_url(url):
    if url is None:
        return '기타'
    elif 'signup' in url:
        return '가입 프로세스'
    elif 'resume' in url:
        return '이력서 작성'
    elif 'apply' in url:
        return '지원 프로세스'
    elif 'jobs' in url or 'api/jobs' in url:
        return '채용공고 탐색'
    elif 'companies' in url or 'api/companies' in url:
        return '기업 탐색'
    elif 'search' in url:
        return '검색'
    elif 'users' in url or '@user' in url:
        return '프로필/설정'
    elif 'notification' in url:
        return '알림'
    elif 'setting' in url or 'email' in url:
        return '설정'
    else:
        return '기타'
